# Data Processing for JD.com Final Project

This notebook builds the cleaned and processed data tables for the JD.com e-commerce operations project.

## Scope for this notebook

This notebook **does**:

- load cleaned raw tables
- build the delivery summary table
- build the order-line fact table
- add inventory and region-based fulfillment features
- build warehouse assignment candidate rows
- save processed tables to `data/processed/`



In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT, RAW_DIR, PROCESSED_DIR


(WindowsPath('C:/Users/Qiang Gao/Desktop/IE5404-JD'),
 WindowsPath('C:/Users/Qiang Gao/Desktop/IE5404-JD/data/raw'),
 WindowsPath('C:/Users/Qiang Gao/Desktop/IE5404-JD/data/processed'))

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print(SRC_DIR)

C:\Users\Qiang Gao\Desktop\IE5404-JD\src


## Imports

Import the Part A data-processing functions from the project package.


In [3]:
from jd_project.data import load_raw_tables
from jd_project.features import (
    build_delivery_summary,
    build_order_line_fact,
    build_inventory_features,
    build_assignment_candidates,
    save_processed_tables,
    run_basic_quality_checks,
)


## Load cleaned input tables


In [4]:
tables = load_raw_tables(RAW_DIR)

orders_df = tables["orders"]
delivery_df = tables["delivery"]
inventory_df = tables["inventory"]
network_df = tables["network"]
users_df = tables["users"]
skus_df = tables["skus"]

for name, df in tables.items():
    print(f"{name:10s} shape = {df.shape}")


orders     shape = (549989, 20)
delivery   shape = (293229, 6)
inventory  shape = (136079, 4)
network    shape = (56, 2)
users      shape = (457298, 10)
skus       shape = (31867, 7)


In [5]:
orders_df.head(3)


,order_id,user_id,sku_id,order_date,order_time,quantity,type,promise,original_unit_price,final_unit_price,direct_discount_per_unit,quantity_discount_per_unit,bundle_discount_per_unit,coupon_discount_per_unit,gift_item,dc_ori,dc_des,discount_total_per_unit,discount_gap,discount_consistency_flag
0,d0cf5cc6db,0abe9ef2ce,581d5b54c1,2018-03-01,2018-03-01 17:14:25,1,2,<NA>,89.0,79.0,0.0,10.0,0.0,0.0,0,4,28,10.0,0.0,1
1,7444318d01,33a9e56257,067b673f2b,2018-03-01,2018-03-01 11:10:40,1,1,2,99.9,53.9,5.0,41.0,0.0,0.0,0,28,28,46.0,0.0,1
2,f973b01694,4ea3cf408f,623d0a582a,2018-03-01,2018-03-01 09:13:26,1,1,2,78.0,58.5,19.5,0.0,0.0,0.0,0,28,28,19.5,0.0,1


In [6]:
delivery_df.head(3)


,package_id,order_id,type,ship_out_time,arr_station_time,arr_time
0,dc3d6d2258,dc3d6d2258,1,2018-03-01 08:00:00,2018-03-01 15:00:00,2018-03-01 18:00:00
1,19802a570c,19802a570c,1,2018-03-01 10:00:00,2018-03-01 15:00:00,2018-03-01 17:00:00
2,e22627af66,e22627af66,1,2018-03-01 11:00:00,2018-03-01 15:00:00,2018-03-01 17:00:00


In [7]:
inventory_df.head(3)


,dc_id,sku_id,date,inventory_available
0,9,50f6f91962,2018-03-01,1
1,9,7f0ddbcdde,2018-03-01,1
2,9,8ad5789d74,2018-03-01,1


In [8]:
network_df.head(3)


,region_id,dc_id
0,2,57
1,2,43
2,2,42


In [9]:
users_df.head(3)


,user_id,user_level,first_order_month,plus,gender,age,marital_status,education,city_level,purchase_power
0,000089d6a6,1,2017-08-01,0,F,26-35,S,3,4,3
1,0000babd1f,1,2018-03-01,0,U,U,U,-1,-1,-1
2,0000bc018b,3,2016-06-01,0,F,>=56,M,3,2,3


In [10]:
skus_df.head(3)


,sku_id,type,brand_id,attribute1,attribute2,activate_date,deactivate_date
0,a234e08c57,1,c3ab4bf4d9,3.0,60.0,NaT,NaT
1,6449e1fd87,1,1d8b4b4c63,2.0,50.0,NaT,NaT
2,09b70fcd83,2,eb7d2a675a,3.0,70.0,NaT,NaT


## Build delivery summary

This table is an order-level proxy aggregated from package-level delivery data.

### Target grain

One row per `order_id`

### Output columns

- `order_id`
- `package_count`
- `first_ship_out_time`
- `first_arr_station_time`
- `final_arrival_time`


In [11]:
delivery_summary_df = build_delivery_summary(delivery_df)
delivery_summary_df.head()


,order_id,package_count,first_ship_out_time,first_arr_station_time,final_arrival_time
0,0000095025,1,2018-03-19 14:00:00,2018-03-20 06:00:00,2018-03-20 10:00:00
1,00000e13eb,1,2018-03-10 10:00:00,2018-03-11 08:00:00,2018-03-11 18:00:00
2,0000132b39,1,2018-03-13 20:00:00,2018-03-14 08:00:00,2018-03-14 15:00:00
3,000064fa67,1,2018-03-02 11:00:00,2018-03-02 16:00:00,2018-03-02 19:00:00
4,0000bde331,1,2018-03-17 16:00:00,2018-03-18 07:00:00,2018-03-18 12:00:00


In [12]:
delivery_summary_df.shape


(293204, 5)

## Build order-line fact table

### Target grain

One row per **orders row** (cleaned `orders` grain). Same `order_id` + `sku_id` can repeat; each row gets `order_line_id` = `{order_id}_{sku_id}_{k}` with `k = 0,1,...` in row order.

### Merges

- `orders + users` on `user_id`
- `orders + skus` on `sku_id`
- `orders + delivery_summary` on `order_id`

### Key derived columns

- `order_line_id`
- `gross_merchandise_value`
- `net_merchandise_value`
- `discount_total_per_unit`
- `discount_rate`
- `is_discounted`
- `remote_fulfillment_flag`
- `order_hour`
- `order_weekday`
- `order_weekend_flag`
- `promise_days`
- `delivered_by_jd_flag`
- `package_count`
- `hours_to_ship`
- `hours_to_delivery`


In [13]:
order_line_fact_df = build_order_line_fact(
    orders_df=orders_df,
    users_df=users_df,
    skus_df=skus_df,
    delivery_summary_df=delivery_summary_df,
)

order_line_fact_df.head()


,order_id,user_id,sku_id,order_date,order_time,quantity,type,promise,original_unit_price,final_unit_price,direct_discount_per_unit,quantity_discount_per_unit,bundle_discount_per_unit,coupon_discount_per_unit,gift_item,dc_ori,dc_des,discount_total_per_unit,discount_gap,discount_consistency_flag,user_level,first_order_month,plus,gender,age,marital_status,education,city_level,purchase_power,type_sku,brand_id,attribute1,attribute2,activate_date,deactivate_date,package_count,first_ship_out_time,first_arr_station_time,final_arrival_time,order_line_id,gross_merchandise_value,net_merchandise_value,discount_rate,is_discounted,remote_fulfillment_flag,order_hour,order_weekday,order_weekend_flag,promise_days,delivered_by_jd_flag,hours_to_ship,hours_to_delivery
0,d0cf5cc6db,0abe9ef2ce,581d5b54c1,2018-03-01,2018-03-01 17:14:25,1,2,<NA>,89.0,79.0,0.0,10.0,0.0,0.0,0,4,28,10.0,0.0,1,1,2013-06-01,0,F,36-45,M,3,2,2,2,198cec62a1,4.0,100.0,NaT,NaT,NaN,NaT,NaT,NaT,d0cf5cc6db_581d5b54c1_0,89.0,79.0,0.11236,1,1,17.0,3.0,0,<NA>,0,NaN,NaN
1,7444318d01,33a9e56257,067b673f2b,2018-03-01,2018-03-01 11:10:40,1,1,2,99.9,53.9,5.0,41.0,0.0,0.0,0,28,28,46.0,0.0,1,1,2016-05-01,0,F,26-35,S,2,2,3,1,9b0d3a5fc6,3.0,60.0,NaT,NaT,1.0,2018-03-01 13:00:00,2018-03-02 08:00:00,2018-03-02 14:00:00,7444318d01_067b673f2b_0,99.9,53.9,0.46046,1,0,11.0,3.0,0,2,1,1.822222,26.822222
2,f973b01694,4ea3cf408f,623d0a582a,2018-03-01,2018-03-01 09:13:26,1,1,2,78.0,58.5,19.5,0.0,0.0,0.0,0,28,28,19.5,0.0,1,2,2013-02-01,0,F,16-25,S,2,2,2,<NA>,<NA>,<NA>,<NA>,NaT,NaT,1.0,2018-03-01 14:00:00,2018-03-02 09:00:00,2018-03-02 13:00:00,f973b01694_623d0a582a_0,78.0,58.5,0.25,1,0,9.0,3.0,0,2,1,4.776111,27.776111
3,8c1cec8d4b,b87cb736cb,fc5289b139,2018-03-01,2018-03-01 21:29:50,1,1,2,61.0,35.0,0.0,26.0,0.0,0.0,0,4,28,26.0,0.0,1,3,2014-08-01,0,F,26-35,M,3,2,2,<NA>,<NA>,<NA>,<NA>,NaT,NaT,1.0,2018-03-02 09:00:00,2018-03-03 08:00:00,2018-03-04 11:00:00,8c1cec8d4b_fc5289b139_0,61.0,35.0,0.42623,1,1,21.0,3.0,0,2,1,11.502778,61.502778
4,d43a33c38a,4829223b6f,623d0a582a,2018-03-01,2018-03-01 19:13:37,1,1,1,78.0,53.0,19.0,0.0,0.0,6.0,0,3,16,25.0,0.0,1,1,2014-10-01,0,F,26-35,M,2,2,4,<NA>,<NA>,<NA>,<NA>,NaT,NaT,1.0,2018-03-01 20:00:00,2018-03-02 07:00:00,2018-03-02 11:00:00,d43a33c38a_623d0a582a_0,78.0,53.0,0.320513,1,1,19.0,3.0,0,1,1,0.773056,15.773056


In [14]:
order_line_fact_df.shape


(549989, 52)

## Inspect key engineered columns in the order-line fact table

Focus on whether the core transaction and fulfillment fields look reasonable before adding inventory features.


In [15]:
order_line_fact_df[
    [
        "order_line_id",
        "order_id",
        "user_id",
        "sku_id",
        "order_date",
        "order_time",
        "quantity",
        "gross_merchandise_value",
        "net_merchandise_value",
        "discount_total_per_unit",
        "discount_rate",
        "is_discounted",
        "dc_ori",
        "dc_des",
        "remote_fulfillment_flag",
        "package_count",
        "hours_to_ship",
        "hours_to_delivery",
    ]
].head(10)


,order_line_id,order_id,user_id,sku_id,order_date,order_time,quantity,gross_merchandise_value,net_merchandise_value,discount_total_per_unit,discount_rate,is_discounted,dc_ori,dc_des,remote_fulfillment_flag,package_count,hours_to_ship,hours_to_delivery
0,d0cf5cc6db_581d5b54c1_0,d0cf5cc6db,0abe9ef2ce,581d5b54c1,2018-03-01,2018-03-01 17:14:25,1,89.0,79.0,10.0,0.11236,1,4,28,1,NaN,NaN,NaN
1,7444318d01_067b673f2b_0,7444318d01,33a9e56257,067b673f2b,2018-03-01,2018-03-01 11:10:40,1,99.9,53.9,46.0,0.46046,1,28,28,0,1.0,1.822222,26.822222
2,f973b01694_623d0a582a_0,f973b01694,4ea3cf408f,623d0a582a,2018-03-01,2018-03-01 09:13:26,1,78.0,58.5,19.5,0.25,1,28,28,0,1.0,4.776111,27.776111
3,8c1cec8d4b_fc5289b139_0,8c1cec8d4b,b87cb736cb,fc5289b139,2018-03-01,2018-03-01 21:29:50,1,61.0,35.0,26.0,0.42623,1,4,28,1,1.0,11.502778,61.502778
4,d43a33c38a_623d0a582a_0,d43a33c38a,4829223b6f,623d0a582a,2018-03-01,2018-03-01 19:13:37,1,78.0,53.0,25.0,0.320513,1,3,16,1,1.0,0.773056,15.773056
5,e0f5386d87_589c2b865b_0,e0f5386d87,0b07cae293,589c2b865b,2018-03-01,2018-03-01 21:09:15,1,79.9,38.9,41.0,0.513141,1,3,16,1,1.0,0.845833,14.845833
6,89286e5fd9_6717b7c979_0,89286e5fd9,79154d0001,6717b7c979,2018-03-01,2018-03-01 22:18:41,1,0.0,0.0,0.0,<NA>,0,3,16,1,1.0,0.688611,14.688611
7,72585b87a6_d829f03a28_0,72585b87a6,d5e8910932,d829f03a28,2018-03-01,2018-03-01 15:28:49,2,159.8,81.8,39.0,0.48811,1,3,16,1,1.0,22.519722,45.519722
8,72585b87a6_5f58bfd286_0,72585b87a6,d5e8910932,5f58bfd286,2018-03-01,2018-03-01 15:28:49,1,79.9,37.9,42.0,0.525657,1,3,16,1,1.0,22.519722,45.519722
9,9c65b6264b_068f4481b3_0,9c65b6264b,2021a86702,068f4481b3,2018-03-01,2018-03-01 00:12:07,1,298.0,208.0,90.0,0.302013,1,3,16,1,1.0,72.798056,83.798056


## Build inventory and regional fulfillment features

This step enriches the order-line fact table with inventory availability and regional warehouse context.

### Added columns

- `destination_region_id`
- `inventory_at_dc_des`
- `inventory_at_dc_ori`
- `num_available_dcs_in_region`
- `any_inventory_in_region`
- `local_available_but_remote_shipped`


In [16]:
order_line_with_inventory_df = build_inventory_features(
    order_line_df=order_line_fact_df,
    inventory_df=inventory_df,
    network_df=network_df,
)

order_line_with_inventory_df.head()


,order_id,user_id,sku_id,order_date,order_time,quantity,type,promise,original_unit_price,final_unit_price,direct_discount_per_unit,quantity_discount_per_unit,bundle_discount_per_unit,coupon_discount_per_unit,gift_item,dc_ori,dc_des,discount_total_per_unit,discount_gap,discount_consistency_flag,user_level,first_order_month,plus,gender,age,marital_status,education,city_level,purchase_power,type_sku,brand_id,attribute1,attribute2,activate_date,deactivate_date,package_count,first_ship_out_time,first_arr_station_time,final_arrival_time,order_line_id,gross_merchandise_value,net_merchandise_value,discount_rate,is_discounted,remote_fulfillment_flag,order_hour,order_weekday,order_weekend_flag,promise_days,delivered_by_jd_flag,hours_to_ship,hours_to_delivery,destination_region_id,inventory_at_dc_des,inventory_at_dc_ori,num_available_dcs_in_region,any_inventory_in_region,local_available_but_remote_shipped
0,d0cf5cc6db,0abe9ef2ce,581d5b54c1,2018-03-01,2018-03-01 17:14:25,1,2,<NA>,89.0,79.0,0.0,10.0,0.0,0.0,0,4,28,10.0,0.0,1,1,2013-06-01,0,F,36-45,M,3,2,2,2,198cec62a1,4.0,100.0,NaT,NaT,NaN,NaT,NaT,NaT,d0cf5cc6db_581d5b54c1_0,89.0,79.0,0.11236,1,1,17.0,3.0,0,<NA>,0,NaN,NaN,4,0,0,0,0,0
1,7444318d01,33a9e56257,067b673f2b,2018-03-01,2018-03-01 11:10:40,1,1,2,99.9,53.9,5.0,41.0,0.0,0.0,0,28,28,46.0,0.0,1,1,2016-05-01,0,F,26-35,S,2,2,3,1,9b0d3a5fc6,3.0,60.0,NaT,NaT,1.0,2018-03-01 13:00:00,2018-03-02 08:00:00,2018-03-02 14:00:00,7444318d01_067b673f2b_0,99.9,53.9,0.46046,1,0,11.0,3.0,0,2,1,1.822222,26.822222,4,0,0,6,1,0
2,f973b01694,4ea3cf408f,623d0a582a,2018-03-01,2018-03-01 09:13:26,1,1,2,78.0,58.5,19.5,0.0,0.0,0.0,0,28,28,19.5,0.0,1,2,2013-02-01,0,F,16-25,S,2,2,2,<NA>,<NA>,<NA>,<NA>,NaT,NaT,1.0,2018-03-01 14:00:00,2018-03-02 09:00:00,2018-03-02 13:00:00,f973b01694_623d0a582a_0,78.0,58.5,0.25,1,0,9.0,3.0,0,2,1,4.776111,27.776111,4,1,1,6,1,0
3,8c1cec8d4b,b87cb736cb,fc5289b139,2018-03-01,2018-03-01 21:29:50,1,1,2,61.0,35.0,0.0,26.0,0.0,0.0,0,4,28,26.0,0.0,1,3,2014-08-01,0,F,26-35,M,3,2,2,<NA>,<NA>,<NA>,<NA>,NaT,NaT,1.0,2018-03-02 09:00:00,2018-03-03 08:00:00,2018-03-04 11:00:00,8c1cec8d4b_fc5289b139_0,61.0,35.0,0.42623,1,1,21.0,3.0,0,2,1,11.502778,61.502778,4,1,1,7,1,1
4,d43a33c38a,4829223b6f,623d0a582a,2018-03-01,2018-03-01 19:13:37,1,1,1,78.0,53.0,19.0,0.0,0.0,6.0,0,3,16,25.0,0.0,1,1,2014-10-01,0,F,26-35,M,2,2,4,<NA>,<NA>,<NA>,<NA>,NaT,NaT,1.0,2018-03-01 20:00:00,2018-03-02 07:00:00,2018-03-02 11:00:00,d43a33c38a_623d0a582a_0,78.0,53.0,0.320513,1,1,19.0,3.0,0,1,1,0.773056,15.773056,<NA>,0,1,0,0,0


In [17]:
order_line_with_inventory_df.shape


(549989, 58)

## Inspect the inventory-based features

This is a useful sanity check before building assignment candidates.


In [18]:
order_line_with_inventory_df[
    [
        "order_line_id",
        "sku_id",
        "order_date",
        "dc_ori",
        "dc_des",
        "destination_region_id",
        "inventory_at_dc_des",
        "inventory_at_dc_ori",
        "num_available_dcs_in_region",
        "any_inventory_in_region",
        "remote_fulfillment_flag",
        "local_available_but_remote_shipped",
    ]
].head(10)


,order_line_id,sku_id,order_date,dc_ori,dc_des,destination_region_id,inventory_at_dc_des,inventory_at_dc_ori,num_available_dcs_in_region,any_inventory_in_region,remote_fulfillment_flag,local_available_but_remote_shipped
0,d0cf5cc6db_581d5b54c1_0,581d5b54c1,2018-03-01,4,28,4,0,0,0,0,1,0
1,7444318d01_067b673f2b_0,067b673f2b,2018-03-01,28,28,4,0,0,6,1,0,0
2,f973b01694_623d0a582a_0,623d0a582a,2018-03-01,28,28,4,1,1,6,1,0,0
3,8c1cec8d4b_fc5289b139_0,fc5289b139,2018-03-01,4,28,4,1,1,7,1,1,1
4,d43a33c38a_623d0a582a_0,623d0a582a,2018-03-01,3,16,<NA>,0,1,0,0,1,0
5,e0f5386d87_589c2b865b_0,589c2b865b,2018-03-01,3,16,<NA>,0,1,0,0,1,0
6,89286e5fd9_6717b7c979_0,6717b7c979,2018-03-01,3,16,<NA>,0,1,0,0,1,0
7,72585b87a6_d829f03a28_0,d829f03a28,2018-03-01,3,16,<NA>,0,1,0,0,1,0
8,72585b87a6_5f58bfd286_0,5f58bfd286,2018-03-01,3,16,<NA>,0,1,0,0,1,0
9,9c65b6264b_068f4481b3_0,068f4481b3,2018-03-01,3,16,<NA>,0,1,0,0,1,0


## Build assignment candidates

This table expands each order line to all candidate DCs in the destination region.

### Target grain

One row per `order_line_id + candidate_dc`

### Included columns

- `order_line_id`
- `order_id`
- `sku_id`
- `user_id` if available
- `order_date`
- `dc_des`
- `dc_ori`
- `destination_region_id`
- `candidate_dc`
- `candidate_in_destination_region_flag`
- `candidate_remote_flag`
- `inventory_available`

### Rules

- only include candidate DCs from the destination region
- do not include capacity features in Part A


In [19]:
assignment_candidates_df = build_assignment_candidates(
    order_line_df=order_line_fact_df,
    inventory_df=inventory_df,
    network_df=network_df,
)

assignment_candidates_df.head()


,order_line_id,order_id,sku_id,user_id,order_date,dc_des,dc_ori,destination_region_id,candidate_dc,candidate_in_destination_region_flag,candidate_remote_flag,inventory_available
0,0000095025_ecc00df0b6_0,0000095025,ecc00df0b6,57648ed1fc,2018-03-19,2,2,2,2,1,0,1
1,0000095025_ecc00df0b6_0,0000095025,ecc00df0b6,57648ed1fc,2018-03-19,2,2,2,6,1,1,0
2,0000095025_ecc00df0b6_0,0000095025,ecc00df0b6,57648ed1fc,2018-03-19,2,2,2,15,1,1,0
3,0000095025_ecc00df0b6_0,0000095025,ecc00df0b6,57648ed1fc,2018-03-19,2,2,2,20,1,1,0
4,0000095025_ecc00df0b6_0,0000095025,ecc00df0b6,57648ed1fc,2018-03-19,2,2,2,42,1,1,0


In [20]:
assignment_candidates_df.shape


(3809107, 12)

## Run basic quality checks

These checks are lightweight and useful before saving the processed outputs.


In [21]:
quality_checks = run_basic_quality_checks(
    delivery_summary_df=delivery_summary_df,
    order_line_fact_df=order_line_fact_df,
    order_line_with_inventory_df=order_line_with_inventory_df,
    assignment_candidates_df=assignment_candidates_df,
)

quality_checks


{'delivery_summary': {'row_count': 293204, 'duplicate_order_id': 0},
 'order_line_fact': {'row_count': 549989, 'duplicate_order_line_id': 0},
 'order_line_with_inventory': {'row_count': 549989,
  'duplicate_order_line_id': 0},
 'assignment_candidates': {'row_count': 3809107,
  'duplicate_order_line_candidate': 0}}

In [22]:
pd.DataFrame(quality_checks).T


,row_count,duplicate_order_id,duplicate_order_line_id,duplicate_order_line_candidate
delivery_summary,293204.0,0.0,NaN,NaN
order_line_fact,549989.0,NaN,0.0,NaN
order_line_with_inventory,549989.0,NaN,0.0,NaN
assignment_candidates,3809107.0,NaN,NaN,0.0


## Additional quick sanity checks

These checks help confirm that the processed tables have the expected grains and no obvious duplication issues.


In [23]:
print(
    "delivery_summary duplicate order_id:",
    delivery_summary_df.duplicated(subset=["order_id"]).sum(),
)

print(
    "order_line_fact duplicate order_line_id:",
    order_line_fact_df.duplicated(subset=["order_line_id"]).sum(),
)

print(
    "order_line_with_inventory duplicate order_line_id:",
    order_line_with_inventory_df.duplicated(subset=["order_line_id"]).sum(),
)

print(
    "assignment_candidates duplicate (order_line_id, candidate_dc):",
    assignment_candidates_df.duplicated(subset=["order_line_id", "candidate_dc"]).sum(),
)


delivery_summary duplicate order_id: 0
order_line_fact duplicate order_line_id: 0
order_line_with_inventory duplicate order_line_id: 0
assignment_candidates duplicate (order_line_id, candidate_dc): 0


In [24]:
assignment_candidates_df["inventory_available"].value_counts(dropna=False)


inventory_available
0    2772180
1    1036927
Name: count, dtype: Int64

## Save processed tables

Save the four processed outputs to `data/processed/`:

- `delivery_summary.csv`
- `order_line_fact.csv`
- `order_line_with_inventory.csv`
- `assignment_candidates.csv`


In [25]:
save_processed_tables(
    output_dir=PROCESSED_DIR,
    delivery_summary_df=delivery_summary_df,
    order_line_fact_df=order_line_fact_df,
    order_line_with_inventory_df=order_line_with_inventory_df,
    assignment_candidates_df=assignment_candidates_df,
)

print("Saved files to:", PROCESSED_DIR)


Saved files to: C:\Users\Qiang Gao\Desktop\IE5404-JD\data\processed


In [26]:
sorted([p.name for p in PROCESSED_DIR.glob("*.csv")])


['assignment_candidates.csv',
 'delivery_summary.csv',
 'order_line_fact.csv',
 'order_line_with_inventory.csv']

## Final table summary

This final summary gives a compact view of the processed outputs created.


In [27]:
summary_df = pd.DataFrame(
    {
        "table_name": [
            "delivery_summary",
            "order_line_fact",
            "order_line_with_inventory",
            "assignment_candidates",
        ],
        "rows": [
            len(delivery_summary_df),
            len(order_line_fact_df),
            len(order_line_with_inventory_df),
            len(assignment_candidates_df),
        ],
        "columns": [
            delivery_summary_df.shape[1],
            order_line_fact_df.shape[1],
            order_line_with_inventory_df.shape[1],
            assignment_candidates_df.shape[1],
        ],
    }
)

summary_df


,table_name,rows,columns
0,delivery_summary,293204,5
1,order_line_fact,549989,52
2,order_line_with_inventory,549989,58
3,assignment_candidates,3809107,12


In [28]:
from jd_project.data import load_raw_tables
from jd_project.features import (
    build_delivery_summary,
    build_order_line_fact,
    build_inventory_features,
    build_assignment_candidates,
    save_processed_tables,
    run_basic_quality_checks,
)